In [ ]:
# Scenario: The Healthcare Policy Navigator
# Background
# You are part of the Healthcare Compliance & Policy Team at a large hospital network.
# New government regulations on patient data privacy and telemedicine practices have just been released.
# The hospital must quickly adapt to ensure compliance and avoid penalties.
# Challenge
# The hospital uploads a PDF of the healthcare regulation into the Policy Navigator (your Gradio + Chroma + LLM app).
# Your task is to:
# - Process the regulation document so the assistant can store and understand it.
# - Ask compliance-related questions about patient rights, telemedicine rules, and penalties.
# - Generate clear, actionable answers that can guide doctors, administrators, and IT staff

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
import gradio as gr

print("Loading LLM...")

# Load PDF
loader = PyPDFLoader("National Telemedicine Privacy Act, 2026.pdf")
pages = loader.load()

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(pages)

# Embed and store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectordb = Chroma.from_documents(chunks, embeddings)

# Load LLM
hf_pipeline = pipeline("text2text-generation", model="google/flan-t5-base", max_new_tokens=256)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# Build RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever(search_kwargs={"k": 3})
)

def answer_query(question):
    return qa_chain.run(question)

# Gradio interface
demo = gr.Interface(
    fn=answer_query,
    inputs=gr.Textbox(label="Ask a compliance question"),
    outputs=gr.Textbox(label="Answer"),
    title="Healthcare Policy Navigator"
)

demo.launch()